# Network Analysis Project: e-mail networks (email-Eu-core)
## Notebook 1: loading the networks + Section 5.1 (RQ1, network structure)

**What this notebook does, in short**
1. Downloads the dataset from SNAP.
2. Creates `load_networks.py`, the **shared loader** every group member uses, so that we all analyse *exactly* the same graphs.
3. Computes the measures for **RQ1: What is the overall structure of the e-mail network, and do departments differ structurally?**
4. Saves a comparison table (CSV + LaTeX) and a degree-distribution figure into the folder `results/`.

**How to use it (Google Colab)**
- Run the cells **from top to bottom**: click ▶ next to each cell, or press `Shift + Enter`.
- Grey cells are code, white cells (like this one) are explanations.
- If a cell shows a red error, copy the error message and send it to the group.

**Tools:** everything is computed with **NetworkX**, one of the tools introduced in the course. Figures for the report can later be made with Gephi.

## Step 1: Download the data

We use the **SNAP email-Eu-core** dataset (https://snap.stanford.edu/data/email-Eu-core.html):
- `email-Eu-core.txt.gz`: one line per (sender, receiver) pair: person *u* sent **at least one** e-mail to person *v*. 1,005 people, 25,571 directed links.
- `email-Eu-core-department-labels.txt.gz`: one line per person: `person_id department_id` (42 departments).

The commands below start with `!`, which means "run this as a terminal command" instead of Python:
- `mkdir -p data` creates a folder called `data`
- `wget -q -P data <url>` downloads a file into that folder (`-q` = quiet)
- `ls data` lists the folder, so we can check that both files are there

We keep the files compressed (`.gz`): Python can read them directly.

In [ ]:
# Create the data folder and download the two SNAP files into it
!mkdir -p data
!wget -q -P data https://snap.stanford.edu/data/email-Eu-core.txt.gz
!wget -q -P data https://snap.stanford.edu/data/email-Eu-core-department-labels.txt.gz

# Check: you should see the two .gz files listed
!ls data

## Step 2: The shared loader (`load_networks.py`)

This cell **writes a Python file** called `load_networks.py` (that is what the first line `%%writefile` does). Every member imports it, so that all sections of the report are based on identical networks.

**Construction rules (these are also described in Section 3 of the report):**
| Rule | Choice | Why |
|---|---|---|
| Nodes | members of the institution | given by the dataset |
| Edges | u → v if u sent ≥ 1 e-mail to v | the dataset only says *whether* someone wrote, not how often |
| Direction | **directed** | "A writes to B" is different from "B writes to A" |
| Weights | **unweighted** | no e-mail counts in this file |
| Self-loops | **removed** | an e-mail to oneself is not communication between people |
| Isolated nodes | **kept** | people who never e-mailed are still members of the institution |
| Node attribute | `dept` = department ID (0–41) | used for comparisons (RQ1) and homophily (RQ3) |

**Department networks:** for each of the **4 largest departments** we take the members of that department and keep only the e-mails where **both sender and receiver** belong to it (in graph terms: the *induced subgraph*). Node IDs stay the same as in the whole network, so results can be linked across sections.

**Main functions:**
- `load_whole()` → the whole institution network
- `largest_departments(G, k)` → IDs of the k largest departments
- `dept_network(G, dept)` → the network of one department
- `load_all()` → a dictionary with all 5 networks: `"whole"`, `"dept<ID>"` × 4

In [ ]:
%%writefile load_networks.py
"""Shared loader for the email-Eu-core networks (used by every group member)."""
import gzip                       # to read the compressed .gz files directly
from collections import Counter   # to count how many people are in each department
from pathlib import Path          # to build file paths that work on every computer

import networkx as nx

DATA = Path(__file__).parent / "data"  # the data folder next to this file
SEED = 42      # fixed random seed: everyone gets the same "random" results
N_DEPTS = 4    # number of largest departments analysed as separate networks


def _open(path):
    """Open a file as text, whether it is compressed (.gz) or not."""
    path = Path(path)
    return gzip.open(path, "rt") if path.suffix == ".gz" else open(path)


def load_whole():
    """Build the whole institution network as a directed graph with department labels."""
    G = nx.DiGraph()  # DiGraph = directed graph (edges have a direction)

    # 1) Add every person as a node, with their department stored as attribute "dept".
    #    Doing this first also keeps people who never sent/received an e-mail (isolates).
    with _open(DATA / "email-Eu-core-department-labels.txt.gz") as f:
        for line in f:
            if line.strip():                       # skip empty lines
                node, dept = map(int, line.split())
                G.add_node(node, dept=dept)

    # 2) Add one directed edge u -> v for every line "u v" (u sent at least one e-mail to v).
    #    A DiGraph keeps at most one edge per ordered pair, so the network is unweighted.
    with _open(DATA / "email-Eu-core.txt.gz") as f:
        for line in f:
            if line.strip():
                u, v = map(int, line.split()[:2])
                G.add_edge(u, v)

    # 3) Remove self-loops (e-mails a person sent to themselves).
    G.remove_edges_from(list(nx.selfloop_edges(G)))
    return G


def largest_departments(G, k=N_DEPTS):
    """Return the IDs of the k largest departments (ties broken by the smaller department ID)."""
    sizes = Counter(d for _, d in G.nodes(data="dept"))  # {department: number of members}
    ranked = sorted(sizes.items(), key=lambda x: (-x[1], x[0]))  # biggest first
    return [d for d, _ in ranked[:k]]


def dept_network(G, dept):
    """Network of one department: its members + only the e-mails among them (induced subgraph)."""
    members = [n for n, d in G.nodes(data="dept") if d == dept]
    return G.subgraph(members).copy()  # .copy() makes it an independent graph


def load_all():
    """Return a dictionary with the whole network and the N_DEPTS largest department networks."""
    G = load_whole()
    nets = {"whole": G}
    for dept in largest_departments(G):
        nets[f"dept{dept}"] = dept_network(G, dept)  # e.g. "dept4" = department with ID 4
    return nets


if __name__ == "__main__":
    # Quick check when the file is run directly: print the size of every network.
    for name, H in load_all().items():
        print(f"{name:7s}  nodes={H.number_of_nodes():5d}  edges={H.number_of_edges():6d}")

## Step 3: Load the networks and check their size

We import the loader and build the 5 networks.

**What to expect:** `whole` must have **1,005 nodes**. Its number of edges is a bit lower than the 25,571 reported by SNAP, because we removed self-loops. The other four lines are the departments, named by their real department ID (e.g. `dept4`).

If the numbers look wrong, stop here and tell the group.

In [ ]:
import importlib
import load_networks
importlib.reload(load_networks)  # makes sure the latest version of the file is used

from load_networks import SEED, load_all

nets = load_all()  # dictionary: name -> network

# Print the size of each network
for name, G in nets.items():
    print(f"{name:7s}  nodes={G.number_of_nodes():5d}  edges={G.number_of_edges():6d}")

## Step 4: Settings and helper functions for RQ1

Here we import the libraries and define two helper functions.

**Why an undirected version?** Clustering, shortest paths and small-world measures are standard for **undirected** graphs. We convert with the **OR rule** (L03, *symmetrisation*): A and B are linked if *at least one* of them e-mailed the other.

**Why the largest connected component (LCC)?** Path lengths are only defined between people who can reach each other. If a network has isolated people or separate pieces, the average distance would be infinite. So path-based measures are computed on the LCC, and we report which share of nodes it contains.

**Why random graphs?** A clustering of 0.3 means nothing by itself: is it high or low? We compare each network with **random graphs of the same size** (same number of nodes and edges, the Erdős–Rényi model G(n, m) from L06). If our network has much more clustering than random graphs but similar path lengths, it is a **small-world** network.

In [ ]:
import math
import random
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

OUT = Path("results")
OUT.mkdir(exist_ok=True)   # folder where tables and figures are saved

N_RANDOM = 10              # how many random graphs to average for the baseline
RUN_OMEGA = True           # compute the small-world coefficient omega (slow)
OMEGA_ON_WHOLE = False     # omega on the whole network can take a very long time;
                           # set to True only if you have time to wait

rng = random.Random(SEED)  # random generator with fixed seed -> reproducible results


def lcc_undirected(G):
    # Undirected version (OR rule) restricted to its largest connected component.
    U = G.to_undirected()                                 # A-B linked if A->B or B->A
    biggest = max(nx.connected_components(U), key=len)    # largest piece of the network
    return U.subgraph(biggest).copy()


def random_baseline(U, n_samples, rng):
    # Average clustering and path length of random graphs with the same n and m as U.
    n, m = U.number_of_nodes(), U.number_of_edges()
    clustering, paths = [], []
    for _ in range(n_samples):
        R = nx.gnm_random_graph(n, m, seed=rng.randint(0, 10**9))   # Erdős–Rényi G(n, m)
        R = R.subgraph(max(nx.connected_components(R), key=len))   # its largest component
        clustering.append(nx.average_clustering(R))
        paths.append(nx.average_shortest_path_length(R))
    return float(np.mean(clustering)), float(np.mean(paths))

## Step 5: Compute all RQ1 measures for the 5 networks

For each network the cell below computes the following (A → B → C: phenomenon → measure → meaning for e-mails):

| Measure | NetworkX function | What it tells us about e-mail communication |
|---|---|---|
| **Density** | `nx.density` | share of all possible "who writes to whom" pairs that actually exist. Depends strongly on network size, so compare with care. |
| **Reciprocity** | `nx.reciprocity` | share of links that are answered: if A writes to B, does B also write to A? High → two-way conversation, low → one-way broadcasting. |
| **Mean degree** | edges / nodes | on average, how many colleagues each person writes to. |
| **Transitivity** and **average clustering** | `nx.transitivity`, `nx.average_clustering` | "my contacts also write to each other": tightly knit working groups. |
| **LCC share** / **largest SCC share** | connected components | how much of the institution is reachable (ignoring / respecting e-mail direction). |
| **Average shortest path** / **diameter** | `nx.average_shortest_path_length`, `nx.diameter` | how many "e-mail steps" separate two people on average / at most: how fast information can travel. |
| **Random baseline** | `nx.gnm_random_graph` | clustering and path length expected by chance for a network of the same size. |
| **Sigma (σ)** | (C / C_rand) / (L / L_rand) | **σ > 1** → small-world: much more clustered than random, but with similarly short paths. |
| **Omega (ω)** | `nx.omega` | **ω ≈ 0** → small-world; **ω < 0** → closer to a regular lattice; **ω > 0** → closer to a random graph. Computed only for the departments by default (very slow on the whole network). |
| **log(n)** | | in random graphs path length grows like log(n): a quick reference value for the average shortest path. |

This cell can take **a few minutes** (mostly ω). Wait until the table appears.

In [ ]:
def structure(name, G):
    # Compute all RQ1 measures for one network and return them as a dictionary.
    U_full = G.to_undirected()   # undirected version of the whole graph (OR rule)
    U = lcc_undirected(G)        # undirected largest connected component

    # Clustering and path length of our network ...
    C = nx.average_clustering(U)
    L = nx.average_shortest_path_length(U)
    # ... and of random graphs of the same size
    C_r, L_r = random_baseline(U, N_RANDOM, rng)
    sigma = (C / C_r) / (L / L_r)   # small-world coefficient sigma

    omega = None                    # left empty when not computed
    if RUN_OMEGA and (name != "whole" or OMEGA_ON_WHOLE):
        print(f"  computing omega for {name} (slow)...")
        # niter / nrand kept small to limit computing time (state this in the report)
        omega = nx.omega(U, niter=2, nrand=2, seed=SEED)

    # Largest strongly connected component: everyone can reach everyone FOLLOWING e-mail direction
    largest_scc = max(nx.strongly_connected_components(G), key=len)

    return {
        "network": name,
        "nodes": G.number_of_nodes(),
        "edges (directed)": G.number_of_edges(),
        "density (directed)": nx.density(G),
        "reciprocity": nx.reciprocity(G),
        "mean in/out degree": G.number_of_edges() / G.number_of_nodes(),
        "transitivity": nx.transitivity(U_full),
        "avg clustering (LCC)": C,
        "LCC share of nodes": U.number_of_nodes() / G.number_of_nodes(),
        "largest SCC share": len(largest_scc) / G.number_of_nodes(),
        "avg shortest path (LCC)": L,
        "diameter (LCC)": nx.diameter(U),
        "random avg clustering": C_r,
        "random avg shortest path": L_r,
        "sigma": sigma,
        "omega": omega,
        "log(n)": math.log(G.number_of_nodes()),
    }


# Run the function on all 5 networks
rows = []
for name, G in nets.items():
    print(f"analysing {name} ...")
    rows.append(structure(name, G))

# Build the comparison table: rows = measures, columns = networks
table = pd.DataFrame(rows).set_index("network").T
pd.set_option("display.float_format", "{:.3f}".format)  # show 3 decimals
table

## Step 6: Save the table for the report

- `results/table_5_1.csv`: opens in Excel / Google Sheets.
- `results/table_5_1.tex`: ready to paste into the LaTeX report (3 decimals; `--` means "not computed").

In the report, keep only the rows we actually discuss and give the table a caption, e.g. *"Table 1: Structural measures of the whole network and the four largest departments."*

In [ ]:
table.to_csv(OUT / "table_5_1.csv")
table.to_latex(OUT / "table_5_1.tex", float_format="%.3f", na_rep="--")
print("Saved:", OUT / "table_5_1.csv", "and", OUT / "table_5_1.tex")

## Step 7: Degree distributions

**Phenomenon:** do most people write to (or receive e-mails from) a similar number of colleagues, or are there a few "hubs" with very many contacts?

**Measure:** the degree distribution P(k) = share of people with exactly k contacts, separately for
- **in-degree**: from how many different colleagues a person *receives* e-mails;
- **out-degree**: to how many different colleagues a person *sends* e-mails.

**Plot:** both axes are logarithmic (log-log). A roughly straight, falling line would suggest a heavy tail (few hubs, many people with few contacts).

⚠️ With ~1,000 nodes we **cannot** reliably claim the network is "scale-free" (L06): describe the shape of the distribution with care.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))   # two plots side by side

for kind, ax in zip(["in", "out"], axes):
    for name, G in nets.items():
        degrees = dict(G.in_degree() if kind == "in" else G.out_degree())
        values = np.array([d for d in degrees.values() if d > 0])  # log scale cannot show 0
        ks, counts = np.unique(values, return_counts=True)          # how many people have degree k
        ax.loglog(ks, counts / counts.sum(), "o", ms=3, alpha=0.7, label=name)
    ax.set_xlabel(f"{kind}-degree k")
    ax.set_ylabel("P(k)")
    ax.set_title(f"{kind}-degree distribution (log-log)")
    ax.legend()

fig.tight_layout()
fig.savefig(OUT / "degree_distribution.png", dpi=200)  # high resolution for the report
plt.show()

## Step 8: Download the results

Run the cell below to download the `results` folder as a zip file to your computer (Colab only). Then upload the table and the figure to our GitHub repo.

In [ ]:
# Zip the results folder and download it (works in Google Colab)
!zip -r -q results.zip results
try:
    from google.colab import files
    files.download("results.zip")
except ImportError:
    print("Not running in Colab: the results are in the 'results' folder.")

## Step 9: Compact table for the report

The full table from Step 5 has some rows we do not discuss (e.g. log(n)). This cell selects the rows used in the report, renames them, orders the columns and saves `results/table_report_5_1.tex`, so the numbers in the report always come directly from the code.

In [ ]:
# Rows used in the report (key in the full table -> label in the report), in order
rows = {
    "nodes": "Nodes",
    "edges (directed)": "Edges (directed)",
    "density (directed)": "Density",
    "reciprocity": "Reciprocity",
    "mean in/out degree": "Mean in/out degree",
    "LCC share of nodes": "LCC share of nodes",
    "largest SCC share": "Largest SCC share",
    "transitivity": "Transitivity",
    "avg clustering (LCC)": "Avg. clustering $C$",
    "random avg clustering": "Random avg. clustering $C_{rand}$",
    "avg shortest path (LCC)": "Avg. shortest path $L$",
    "random avg shortest path": "Random avg. shortest path $L_{rand}$",
    "diameter (LCC)": "Diameter",
    "sigma": "$\\sigma$",
    "omega": "$\\omega$",
}

report = table.loc[list(rows)].rename(index=rows)   # keep and rename the selected rows
report.columns = ["Whole" if c == "whole" else c.replace("dept", "Dept ") for c in report.columns]

report.to_latex(
    OUT / "table_report_5_1.tex",
    float_format="%.3f",   # 3 decimals
    na_rep="--",           # omega not computed for the whole network
    escape=False,          # keep the LaTeX symbols ($\sigma$ ...)
)
print(report)
print("\nSaved:", OUT / "table_report_5_1.tex")

## Results so far (RQ1: network structure)

**Networks analysed**
| Network | Nodes | Directed edges |
|---|---|---|
| Whole institution | 1,005 | 24,929 (642 self-loops removed) |
| Dept 4 | 109 | 1,167 |
| Dept 14 | 92 | 1,506 |
| Dept 1 | 65 | 502 |
| Dept 21 | 61 | 601 |

Check: our Dept 14 has exactly 1,506 edges, like SNAP's own "Dept3" file → our way of building department networks matches SNAP's.

### 1. The whole network is a small-world network (strong result)
| | Our network | Random graph, same size |
|---|---|---|
| Avg. clustering | **0.407** | 0.033 |
| Avg. shortest path | **2.59** | 2.30 |

- Clustering is ~**12× higher** than random, while paths are almost as short → **σ = 10.8** (> 1 = small-world).
- **Meaning for e-mail:** people work in tightly knit groups ("my contacts also write to each other"), yet any two members are on average **less than 3 e-mail steps** apart. Diameter = 7. Information can travel fast across the whole institution.

### 2. Communication is mostly two-way
- **Reciprocity = 0.71**: 71% of links go both ways → e-mail works as *conversation*, not one-way broadcasting.
- Highest in Dept 21 (0.82).

### 3. Reachability
- Ignoring direction: **98%** of members are in one connected component.
- Following e-mail direction: **80%** can reach everyone (largest SCC). The other ~20% only send to, or only receive from, the core.

### 4. Departments
| | Dept 4 | Dept 14 | Dept 1 | Dept 21 |
|---|---|---|---|---|
| Avg. clustering | 0.47 | 0.52 | 0.59 | 0.66 |
| σ | 2.88 | 2.00 | 2.08 | 1.78 |
| ω | −0.03 | −0.00 | **−0.15** | 0.00 |
| LCC share | 0.93 | 0.97 | **0.86** | **0.75** |

- All four departments are **small-world** too (σ > 1; ω ≈ 0 for Depts 4, 14, 21).
- **Dept 1** (ω = −0.15) leans towards a **lattice**: local, highly clustered groups with longer chains between them.
- ⭐ **Interesting for RQ3:** in **Dept 21, 25%** and in **Dept 1, 14%** of members exchange **no e-mail with any colleague of their own department** → part of their communication happens *across* departments.

### 5. Degree distribution
- Whole network: **heavy-tailed**. Most people correspond with a few dozen colleagues; a handful with **100–300+**.
- These highly connected people are candidates for the central roles in **RQ2**.
- We do **not** claim "scale-free": ~1,000 nodes are too few for a reliable power-law fit.
- Department distributions are too small to show a clear shape.

### Cautions (written in the report)
- **Do not compare raw density** across networks: departments look denser (0.10–0.18 vs 0.025) mainly because they are smaller.
- **Do not compare σ across sizes**: σ grows with network size, so departments are compared with **ω**.
- ω was computed with light settings (`niter=2, nrand=2`) and **not** for the whole network (too slow): mentioned in the Critique.

### Status
- Section 5.1, Conclusion 6.1 and Critique 7.1 are written in `report/main.tex` (Table 1 + Figure 1).